# **Experiment Notebook**



---
## Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT1",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 72.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT1/data


---
## Student Information

In [2]:
student_name = "SUSHRUTA GANGADHAR PATIL"
student_id = "26273312"

In [3]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [4]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [5]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import altair as alt

In [6]:
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import root_mean_squared_error as rmse

---
## A. Experiment Description

In [7]:
# DO NOT MODIFY THE CODE IN THIS CELL
experiment_id = "3"
print_tile(size="h1", key='experiment_id', value=experiment_id)

In [8]:
experiment_hypothesis = """

The hypothesis is that KNN Regression will outperform both Linear Regression ($27,162.40) and
ElasticNet ($27,872.71) as it does not function on the linear relationship between the features
and the target variable. It predicts based on the actual prices of the most similar cars in the
training data which may capture non-linear pricing patterns that linear models cannot.

"""

In [9]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='experiment_hypothesis', value=experiment_hypothesis)

In [10]:
experiment_expectations = """
I expect a smaller n_neighbors under 12 but above 2 or 3 to perform better since car pricing is quite
specific. A car should be grouped with other cars that are most similar in price to cars with very
similar features rather than a broad neighbourhood of cars.

For p, p=1 uses Manhattan distance and p=2 uses Euclidean distance.
I expect p=2 to perform better since our features are continuous and numeric after scaling, making
Euclidean distance more appropriate.

I also expect KNN to be more sensitive to the time-based split present across data sets than linear
models as it relies on finding similar cars in training data but the validation cars are from a
different time period.

"""

In [11]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='experiment_expectations', value=experiment_expectations)

---
## B. Feature Selection


In [12]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load data
try:
  X_train = pd.read_csv(at.folder_path / 'X_train.csv')
  y_train = pd.read_csv(at.folder_path / 'y_train.csv')

  X_val = pd.read_csv(at.folder_path / 'X_val.csv')
  y_val = pd.read_csv(at.folder_path / 'y_val.csv')

  X_test = pd.read_csv(at.folder_path / 'X_test.csv')
  y_test = pd.read_csv(at.folder_path / 'y_test.csv')
except Exception as e:
  print(e)

In [13]:
feature_selection_explanations = """
Using the same 6 features as Experiments 1 and 2:

1. kilometres_driven        - how far the car has been driven
2. manufacturing_year       - the year the car was manufactured
3. engine_cylinders         - number of engine cylinders
4. brand_bodytype_encoded   - smoothed mean encoded combination
                              of brand and body type
5. body_drive_encoded       - smoothed mean encoded combination
                              of body type and drive type
6. transmission_type_Manual - binary flag, 1 = Manual, 0 = Automatic

Keeping the same features as previous experiments so the comparison is fair. Any difference in RMSE
comes from the algorithm not the features. It is also true that KNN is sensitive to feature scaling
since it calculates distances between records.
All our features are already scaled from the Preparation notebook so KNN can work with them directly.

"""

In [14]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## C. Train Machine Learning Model

### C.1 Import Algorithm



In [15]:
from sklearn.neighbors import KNeighborsRegressor

In [16]:
algorithm_selection_explanations = """

KNN Regression is a non-parametric algorithm that predicts the price of a car by looking at the k
most similar cars in the training data and averaging their prices. Unlike Linear Regression and
ElasticNet which fit a mathematical equation to the data, KNN makes no assumptions about the
relationship between features and price.

I feel it is a good third experiment because it takes a completely different approach to the previous
two models. If there is any non-linear relationship between features and price then KNN should be able
to capture those patterns.

"""

In [17]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='algorithm_selection_explanations', value=algorithm_selection_explanations)

### C.2 Set Hyperparameters


In [18]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    "n_neighbors": randint(1, 50),
    "p": [1, 2]
}

random_search = RandomizedSearchCV(
    KNeighborsRegressor(),
    param_distributions=param_dist,
    n_iter=50,
    scoring="neg_root_mean_squared_error",
    cv=5,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print(f"Best n_neighbors: {random_search.best_params_['n_neighbors']}")
print(f"Best p          : {random_search.best_params_['p']}")
print(f"Best CV RMSE    : ${-random_search.best_score_:,.2f}")

Best n_neighbors: 6
Best p          : 2
Best CV RMSE    : $15,829.22


In [19]:
n_range = range(1, 20)
train_scores = []
val_scores = []

for n in n_range:
    knn = KNeighborsRegressor(n_neighbors=n, p=2)
    knn.fit(X_train, y_train)
    train_scores.append(rmse(y_train, knn.predict(X_train)))
    val_scores.append(rmse(y_val, knn.predict(X_val)))

point_df = pd.DataFrame({
    "n_neighbors": list(n_range),
    "train_rmse": train_scores,
    "val_rmse": val_scores
})

alt.Chart(point_df.melt("n_neighbors", var_name="split", value_name="rmse")).mark_line(point=True).encode(
    alt.X("n_neighbors:Q", title="Number of Neighbours"),
    alt.Y("rmse:Q", title="RMSE ($)", scale=alt.Scale(zero=False)),
    alt.Color("split:N"),
    tooltip=["n_neighbors", "split", "rmse"]
).properties(title="KNN - Training vs Validation RMSE by n_neighbors", width=600, height=300).interactive()

alt.Chart(...)

In [20]:
print(point_df.to_string(index=False))

 n_neighbors   train_rmse     val_rmse
           1    18.437094 22634.729253
           2  9402.461034 22741.349266
           3 12188.541747 23020.735764
           4 13175.109511 23315.127246
           5 14140.076537 23708.478556
           6 14944.825452 23677.670087
           7 15166.789823 23676.158740
           8 15584.987301 24065.150157
           9 15981.557110 24155.847421
          10 16116.066849 24359.013840
          11 16460.031784 24422.963011
          12 16656.255927 24595.964165
          13 16943.030541 24629.394718
          14 17184.851391 24785.842491
          15 17422.786321 24896.968954
          16 17624.889971 25041.020873
          17 17746.226239 25030.086125
          18 17913.476301 25105.866069
          19 18038.229524 25236.247913


In [21]:
hyperparameters_selection_explanations = """

Two hyperparameters n_value and p value are tuned for KNN:

n_neighbors is about how many similar featured cars the model should consider when making a prediction.
A small value means the model looks at very few but highly similar cars. A large value means a
reduction in noise but may also include cars that are not really similar.

p value controls the distance metric.
p=1 uses Manhattan distance which is;
distance = |x1-y1| + |x2-y2| + ... + |xn-yn|

p=2 uses Euclidean distance which is;
distance = squareroot((x1-y1)^2 + (x2-y2)^2 + ... + (xn-yn)^2)

I first used RandomizedSearchCV with 50 iterations searching n_neighbors from 1 to 50 and both p
values with 5 fold cross validation. The best combination found was n_neighbors= 6 and p=2
with a CV RMSE of $14944.82.
To validate this I plotted training and validation RMSE for n_neighbors from 1 to 20 with p=2.
At n=1 the training RMSE was nearly zero at $18.44. This is how KNN with n=1 behaves. When evaluated
on the same training data, every car's nearest neighbour is itself so the model predicts its own
price almost exactly resulting to RMSE so close to 0. This is a classic overfitting and this value
should not be considered as it wont be generalised and the performance on the unseen data would be bad.

Looking at the plot n=7 showed a more balanced result with training RMSE of $15166.78 and validation
RMSE of $23676.16. n=7 was chosen as the final value since it shows a healthier balance between
training and validation performance without memorising the training data.

"""

In [22]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='hyperparameters_selection_explanations', value=hyperparameters_selection_explanations)

### C.3 Fit Model

In [23]:
best_model = KNeighborsRegressor(
    n_neighbors=random_search.best_params_["n_neighbors"],
    p=random_search.best_params_["p"]
)
best_model.fit(X_train, y_train)
print("Model trained successfully")

Model trained successfully


---
## D. Model Evaluation

### D.1 Model Technical Performance

In [24]:
best_model = KNeighborsRegressor(n_neighbors=7, p=2)
best_model.fit(X_train, y_train)

train_rmse = rmse(y_train, best_model.predict(X_train))
val_rmse = rmse(y_val, best_model.predict(X_val))

print(f"n_neighbors=7, p=2")
print(f"Training RMSE  : ${train_rmse:,.2f}")
print(f"Validation RMSE: ${val_rmse:,.2f}")

n_neighbors=7, p=2
Training RMSE  : $15,166.79
Validation RMSE: $23,676.16


In [25]:
results_df = pd.DataFrame([{"model": "Baseline", "train_rmse": 26155., "val_rmse": 37291.73},
    {"model": "Linear Regression", "train_rmse": 20071.20, "val_rmse": 27162.40},
    {"model": "ElasticNet", "train_rmse": 20204.74, "val_rmse": 27872.71},
    {"model": "KNN", "train_rmse": round(train_rmse, 2), "val_rmse": round(val_rmse, 2)}
])

order = ["Baseline", "Linear Regression", "ElasticNet", "KNN"]

alt.Chart(results_df.melt("model", var_name="split", value_name="rmse")).mark_bar().encode(
    alt.X("model:N", title="", sort=order, axis=alt.Axis(labelAngle=-15)),
    alt.Y("rmse:Q", title="RMSE ($)"),
    alt.Color("split:N", legend=alt.Legend(title="Split")),
    alt.Column("split:N", sort=order, title=""),
    tooltip=["model", "split", "rmse"]
).properties(
    title=alt.TitleParams("All Models RMSE Comparison", anchor="middle"),
    width=200,
    height=300
).interactive()

alt.Chart(...)

In [26]:
model_performance_explanations = """

KNN Regression was trained on final hyperparameters:
n_neighbors : 7
p           : 2 (Euclidean distance)

Final results:
Baseline          — Train RMSE: $26,155.88 | Val RMSE: $37,417.14
Linear Regression — Train RMSE: $20,062.44 | Val RMSE: $27,170.19
ElasticNet        — Train RMSE: $20,195.94 | Val RMSE: $27,891.50
KNN               — Train RMSE: $15,162.38 | Val RMSE: $23,770.21

Full consistency comparison across all models:
Price Range  | Linear Reg   | ElasticNet   | KNN
<$15k        | $11,530.64   | $10,769.80   | $15,661.17   <- ElasticNet better
$15k-$30k    | $6,279.20    | $5,575.11    | $4,361.37    <- KNN better
$30k-$50k    | $11,443.52   | $11,227.32   | $8,588.60    <- KNN better
$50k-$100k   | $25,540.49   | $26,789.76   | $22,667.82   <- KNN better
>$100k       | $130,738.91  | $134,559.90  | $116,056.90  <- KNN better

KNN is the most accurate and consistent model across all price ranges except <$15k which only had
4 records so is not reliable. Rest all price ranges, we can see that KNN has outperformed the previous
two models significantly.
But still the error on price above $50k is not being handled. This proves that, the problem is with
the data, how there were not enough data to train on the luxury cars, hence bad prediction.

Though this is not a deployable model due to the time-split, non availability of data on luxury cars,
but as an overall KNN performance is better than other two, the test data will be run on KNN to check
its overall and detailed price wise split RMSE to bring in final conclusions on the experiments and
decision on deployment.

"""

In [27]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='model_performance_explanations', value=model_performance_explanations)

In [28]:
# Test data evaluation using KNN
best_model = KNeighborsRegressor(n_neighbors=7, p=2)
best_model.fit(X_train, y_train)
y_test_pred = best_model.predict(X_test)

print(f"KNN Final Test RMSE: ${rmse(y_test, y_test_pred):,.2f}")
print()

for label, lo, hi in [("<$15k",0,15000),("$15k-$30k",15000,30000),("$30k-$50k",30000,50000),("$50k-$100k",50000,100000),(">$100k",100000,9e9)]:
    mask = (y_test.values >= lo) & (y_test.values < hi)
    if mask.sum() == 0: continue
    r = rmse(y_test.values[mask], y_test_pred[mask])
    print(f"{label:<12} | {mask.sum():>5} records | ${r:>10,.2f}")

KNN Final Test RMSE: $40,439.59

$15k-$30k    |   191 records | $  5,242.39
$30k-$50k    |   841 records | $ 11,783.42
$50k-$100k   |   850 records | $ 30,631.87
>$100k       |   156 records | $124,377.38


### D.2 Business Impact from Current Model Performance


In [29]:
business_impacts_explanations = """

KNN outperformed all other models with a validation RMSE of $23,770 and won in 4 out of 5 price
segments. It is the best model when it comes to $15k-$50k price range with RMSE of $4,361 in the
$15k-$30k range and $8,589 in the $30k-$50k range.

When KNN was used on test data, RMSE came in at $40,439.59 which is significantly higher than
validation. I expected this as the test set has 850 records in the $50k-$100k range and 156 above
$100k, far more expensive cars than validation, and these are mostly NEW and DEMO vehicles that
the model was never trained on.

Deployment recommendation: NOT RECOMMENDED.

Despite KNN being the best performing model, deployment is not recommended for the following reasons:
1. Model was trained on pre-2017 manufactured models data while test listings are from 2022-2023.
The prices could have changed significantly in that period due to reasons like inflation, low supply
during covid etc.
2. Test RMSE of $40,439 is significantly higher than validation RMSE of $23,770, a gap of $16,669
all because of the huge count of NEW and DEMO type cars which should be handled differently than the
USED cars.
3. Insufficient luxury car records made the model suffer when predicting cars above $50k. If similar
data is not put in training for the model to understand trends, it could seriously mislead buyers
of luxury cars.

All three models share these same limitations so no model from this experiment is production ready.
Once the above requirements are taken care of, once more the RMSE needs to be calculated and to be decided
on the model and if it is fit for deployment.

"""

In [30]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='business_impacts_explanations', value=business_impacts_explanations)

## E. Conclusion

In [31]:
experiment_outcome = "Hypothesis confirmed"

In [32]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h2", key='experiment_outcomes_explanations', value=experiment_outcome)

In [33]:
experiment_results_explanations = """
The hypothesis was confirmed. KNN outperformed both LinearRegression ($27,170.19) and ElasticNet
($27,891.50) with a validation RMSE of $23,770.21. The model also has lowest RMSE from all price
splits except under $15k range due to unavailability of more data.

Full comparison across all models:
Price Range  | Linear Reg   | ElasticNet   | KNN
<$15k        | $11,530.64   | $10,769.80   | $15,661.17
$15k-$30k    | $6,279.20    | $5,575.11    | $4,361.37
$30k-$50k    | $11,443.52   | $11,227.32   | $8,588.60
$50k-$100k   | $25,540.49   | $26,789.76   | $22,667.82
>$100k       | $130,738.91  | $134,559.90  | $116,056.90

All three models struggle above $50k. This is a data limitation and not a modelling failure, there
are simply not enough luxury car records in training for any model to price them accurately.

Overall progression across all experiments:
Baseline          : $37,417.14
Linear Regression : $27,170.19
ElasticNet        : $27,891.50
KNN               : $23,770.21

"""

In [34]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h2", key='experiment_results_explanations', value=experiment_results_explanations)

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=fbbefce8-41ae-47c6-bc64-96decd566c0b' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>